In [1]:
model = "llama3.2:1b"

#### Task 1: Create a Simple Chain for Summarization


**Objective:**

Build a LangChain chain that can summarize a given text.

**Task Description:**

- Create a llm chain using with a Ollama model.
- Define a prompt template for summarization. The summary should be only one sentence.
- Run the chain with a sample text and print the summary.
- Add model output streaming.
- Run the chain with streaming with a sample text and print the summary.


**Hint: The five messages in LangChain:**


- SystemMessage: corresponds to system role
- HumanMessage: corresponds to user role
- AIMessage: corresponds to assistant role
- AIMessageChunk: corresponds to assistant role, used for streaming responses
- ToolMessage: corresponds to tool role


[More Information](https://python.langchain.com/docs/concepts/messages/)

**Useful links:**

- [How To Prompt Template 1](https://python.langchain.com/v0.2/docs/tutorials/extraction/#the-extractor)
- [How To Prompt Template 2](https://python.langchain.com/v0.2/api_reference/core/prompts/langchain_core.prompts.chat.ChatPromptTemplate.html#langchain_core.prompts.chat.ChatPromptTemplate)
- [How To LCEL Chains 1](https://python.langchain.com/v0.2/docs/concepts/#langchain-expression-language-lcel)
- [How To LCEL Chains 2](https://python.langchain.com/v0.2/docs/versions/migrating_chains/llm_chain/#lcel)
- [How To Chain Streaming](https://python.langchain.com/v0.2/docs/concepts/#streaming)

In [2]:
from langchain_ollama.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

# Load the Ollama model
llm = ChatOllama(model=model)

# ADD HERE YOUR CODE
# Define the prompt template
summarization_prompt = ChatPromptTemplate([
    ("system", "You are a helpful assistant. Summarize the text given by the user in exactly one sentence. Return only the summary."),
    ("human", "{text}"),
])

# ADD HERE YOUR CODE
# Create the LLMChain
summarization_chain = summarization_prompt | llm

# Sample text
text = """Over the last decade, deep learning has evolved massively to process and generate unstructured data like text, images, and video. 
These advanced AI models have gained popularity in various industries, and include large language models (LLMs). 
There is currently a significant level of fanfare in both the media and the industry surrounding AI,
and there’s a fair case to be made that Artificial Intelligence (AI), with these advancements,
is about to have a wide-ranging and major impact on businesses, societies, and individuals alike.
This is driven by numerous factors, including advancements in technology, high-profile applications, 
and the potential for transfor- mative impacts across multiple sectors."""

# ADD HERE YOUR CODE
# Invoke the chain
summary = summarization_chain.invoke({"text": text})
print(summary.content)

Deep learning has revolutionized AI processing and generation of unstructured data like text, images, and video, with significant advancements in recent years driving its widespread impact on various industries.


In [3]:
# Stream the chain output
for chunk in summarization_chain.stream({"text": text}):
    print(chunk.content, end="", flush=True)

Deep

 learning

 has

 undergone

 significant

 evolution

... (22 outputs left out)

 aspects

 of

 society

.

#### Task 2: Chain with Tool Usage (Simple Math Tool)

**Objective:**

Create a LangChain chain that uses a simple math tool to perform calculations.

**Task Description:**

- Define a function as tool which multiplies two integer values and return the result.
- Create a chain
- Print the result of the calculation.

**Useful links:**

- [How To Tools](https://python.langchain.com/v0.2/docs/how_to/tools_chain/#create-a-tool)
- [How To Tools in Chains](https://python.langchain.com/v0.2/docs/how_to/tools_chain/#chains)
- [How To Tool Calling](https://python.langchain.com/v0.2/docs/concepts/#functiontool-calling)
- [How To Chain and Call Tools with Ollama](https://python.langchain.com/v0.2/docs/integrations/chat/ollama/)

In [4]:
from langchain_core.tools import tool

# ADD HERE YOUR CODE
# Create custom tool
@tool
def multiply(first_int: int, second_int: int) -> int:
    """Multiply two integers together."""
    return first_int * second_int


print(multiply.name)
print(multiply.description)
print(multiply.args) # -> definition of tool arguments

# Invoke custom tool
multiply.invoke({"first_int": 4, "second_int": 5})

multiply
Multiply two integers together.
{'first_int': {'title': 'First Int', 'type': 'integer'}, 'second_int': {'title': 'Second Int', 'type': 'integer'}}


20

In [5]:
# Load the Ollama model
llm = ChatOllama(model=model)

# ADD HERE YOUR CODE
# Use bind_tools to pass the definition of our tool in as part of each call to the model, so that the model can invoke the tool
llm_with_tools = llm.bind_tools([multiply])

# When the model invokes the tool, this will show up in the AIMessage.tool_calls attribute of the output -> extract tool parameters from input text
msg = llm_with_tools.invoke("whats 5 times forty two")
print(msg.tool_calls)

[{'name': 'multiply', 'args': {'first_int': '42', 'second_int': '5'}, 'id': '780bebeb-8b06-4814-87f8-17050d84e772', 'type': 'tool_call'}]


In [6]:
# ADD HERE YOUR CODE
# Create the chain: pass the extracte tool parameters from the input text to the tool -> extract the arguments of the first tool_call
chain_with_tools = llm_with_tools | (lambda msg: msg.tool_calls[0]["args"]) | multiply

# Run chain
chain_with_tools.invoke("whats 5 times forty two")

210

#### Task 3: Agent with Tool Usage (Two Tools)

**Objective:** 

Create a LangChain agent that uses two tools to perform tasks.

**Task Description:**

- Define the system prompt.
- Define tools.
- Create an Agent using the Ollama model, system-prompt and tools.
- Run the agent with a prompt that requires one or both tools.
- Observe how the agent uses the tools to complete the task.


**Useful links:**

- [How To](https://docs.langchain.com/oss/javascript/langchain/agents)

In [7]:
from langchain.agents import create_agent

# Load the Ollama model
# llama3.2:1b is too small for multi-step tool calling (it prints the tool schema as text or invents arguments).
# qwen2.5:3b calls the tools reliably, temperature=0 makes the result reproducible.
# Download once with: ollama pull qwen2.5:3b
agent_model = "qwen2.5:3b"
llm = ChatOllama(model=agent_model, temperature=0)

# ADD HERE YOUR CODE
agent_prompt = """You are a helpful math assistant.
Use the provided tools for every calculation instead of calculating yourself.
Call only one tool at a time and wait for its result.
If a task needs several steps, use the result of the previous tool as input for the next tool.
Answer with the final result."""

# Custom math tools
@tool
def add(first_int: int, second_int: int) -> int:
    """Add two integers."""
    return first_int + second_int


@tool
def exponentiate(base: int, exponent: int) -> int:
    """Exponentiate the base to the exponent power."""
    return base**exponent


tools = [add, exponentiate]

In [8]:
# ADD HERE YOUR CODE
# Construct the agent
agent_with_tools = create_agent(model=llm, tools=tools, system_prompt=agent_prompt)

In [9]:
response = agent_with_tools.invoke(
    {"messages": [{"role": "user", "content": "First take 3 to the power of five and afterwards add 12?"}]}
)

# Show every step of the agent: tool calls, tool results and the final answer
for message in response["messages"]:
    if message.type == "ai" and message.tool_calls:
        for tool_call in message.tool_calls:
            print(f"[AI -> Tool call] {tool_call['name']}({tool_call['args']})")
    elif message.type == "tool":
        print(f"[Tool result] {message.name}: {message.content}")
    else:
        print(f"[{message.type}] {message.content}")

[human] First take 3 to the power of five and afterwards add 12?
[AI -> Tool call] exponentiate({'base': 3, 'exponent': 5})
[AI -> Tool call] add({'first_int': 12, 'second_int': 'result_of_previous_computation'})
[Tool result] exponentiate: 243
[Tool result] add: Error invoking tool 'add' with kwargs {'first_int': 12, 'second_int': 'result_of_previous_computation'} with error:
 second_int: Input should be a valid integer, unable to parse string as an integer
 Please fix the error and try again.
[AI -> Tool call] add({'first_int': 12, 'second_int': 243})
[Tool result] add: 255
[ai] 255


**Beobachtung (Task 3):**

Erwartetes Ergebnis: 3⁵ + 12 = 243 + 12 = **255**.

- Der Agent ruft zuerst `exponentiate(3, 5)` auf und erhält **243**.
- Gleichzeitig ruft er `add` schon auf, bevor das Ergebnis da ist, und setzt dabei einen Platzhalter-Text statt einer Zahl ein. Das Tool lehnt das ab, weil die Argumente per Type-Hint (`int`) validiert werden.
- Die Fehlermeldung geht als `ToolMessage` zurück an das Modell. Der Agent korrigiert sich selbst und ruft `add(12, 243)` auf, das Ergebnis ist **255**.

Das ist der Kern eines Agents: Das LLM entscheidet in einer Schleife selbst, welches Tool es aufruft, sieht das Ergebnis (oder den Fehler) und plant den nächsten Schritt.

**Modellwahl:** Mit `llama3.2:1b` funktionierte der Agent nicht. Das Modell gab das Tool-Schema als Text aus oder erfand Argumente, und das Ergebnis war falsch (z. B. 16). Auch `llama3.2:3b` rechnete die Addition lieber selbst, statt das Tool zu benutzen. `qwen2.5:3b` mit `temperature=0` nutzt beide Tools zuverlässig.